# Raw data → SpatialData
Converts raw MERSCOPE outputs into a SpatialData `.zarr` store for SegTraQ.

**Inputs:** `transcripts.parquet` + `nucleus_boundaries.parquet` per sample (from the two data-prep notebooks)  
**Outputs:** `adata.h5ad` + `raw_spatialdata.zarr` per sample  
**Config:** fill in `INPUT_DIR` in the Configuration cell below

In [ ]:
def raw_to_adata(sample, folder):
    # Builds a cell × gene AnnData from raw transcripts + precomputed nucleus boundaries.
    # Transcripts not assigned to any cell (EntityID=NaN) and cells with <2 transcripts are dropped.
    import pandas as pd
    from shapely.geometry import MultiPoint
    import anndata as ad

    transcripts = pd.read_parquet(folder / sample / "transcripts.parquet")
    boundaries = pd.read_parquet(folder / sample / "nucleus_boundaries.parquet")

    transcripts = transcripts[transcripts['EntityID'].notna()]
    cell_counts = transcripts['EntityID'].value_counts()
    valid_cells = cell_counts[cell_counts >= 2].index
    transcripts = transcripts[transcripts['EntityID'].isin(valid_cells)]

    cell_gene_matrix = pd.crosstab(transcripts['EntityID'], transcripts['gene'])

    # Centroids come from nucleus_boundaries (precomputed from the TIFF mask)
    merged_cell_gene_matrix = boundaries.join(cell_gene_matrix, on='EntityID')

    obs = merged_cell_gene_matrix[['centroid_x', 'centroid_y']]
    X = merged_cell_gene_matrix.drop(columns=['centroid_x', 'centroid_y', 'geometry'])
    X = X.fillna(0)
    var = pd.DataFrame(index=X.columns)

    adata = ad.AnnData(X=X, obs=obs, var=var)

    # SegTraQ requires cell_id column and region as a category in obs
    adata.obs["cell_id"] = adata.obs.index.astype(int)
    adata.obs["region"] = "cell_boundaries"
    adata.obs["region"] = adata.obs["region"].astype("category")
    adata.obsm['spatial'] = obs[['centroid_x', 'centroid_y']].values

    adata.write_h5ad(folder / sample / "adata.h5ad")

In [ ]:
import os
from pathlib import Path

# --- CONFIGURATION: fill in paths before running ---
INPUT_DIR = Path("")  # folder containing per-sample subdirs with transcripts and boundaries

# Step 1: build AnnData for each sample
for folder_name in os.listdir(INPUT_DIR):
    print(folder_name)
    raw_to_adata(folder_name, INPUT_DIR)


In [ ]:
def adata_to_spatialdata(sample, input_folder):
    # Wraps AnnData + transcripts + boundary shapes into a SpatialData .zarr store.
    # adata obs_names, shapes index, and transcripts cell_id must all be aligned as strings.
    import spatialdata as sd
    import anndata as ad
    import pandas as pd
    import geopandas as gpd

    adata = ad.read_h5ad(f'{input_folder}/{sample}/adata.h5ad')
    transcripts_df = pd.read_parquet(f'{input_folder}/{sample}/transcripts.parquet')
    cell_shapes_gdf = gpd.read_parquet(f'{input_folder}/{sample}/nucleus_boundaries.parquet')

    cell_shapes_gdf["cell_id"] = cell_shapes_gdf["EntityID"].astype(str)
    cell_shapes_gdf['EntityID'] = cell_shapes_gdf.set_index('EntityID').values.astype(str)

    # Rename columns to SpatialData convention
    transcripts_df = transcripts_df.rename(columns={
        "gene": "feature_name",
        "global_x": "x",
        "global_y": "y",
        "global_z": "z"
    })
    transcripts_df["feature_name"] = transcripts_df["feature_name"].astype("category")
    transcripts_df['cell_id'] = transcripts_df['EntityID'].astype(str)
    transcripts_df["EntityID"] = transcripts_df["EntityID"].astype(str)

    # Align obs_names with cell_shapes_gdf index (required by SpatialData TableModel)
    adata.obs["cell_id"] = cell_shapes_gdf.cell_id.values.astype(str)
    cell_shapes_gdf.index = cell_shapes_gdf.cell_id.values.astype(str)
    adata.obs_names = adata.obs["cell_id"]
    cell_shapes_gdf = cell_shapes_gdf.set_index("cell_id")
    adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
    adata.obs_names = adata.obs["cell_id"]

    sdata = sd.SpatialData(
        points={"transcripts": sd.models.PointsModel.parse(transcripts_df)},
        shapes={"cell_boundaries": sd.models.ShapesModel.parse(cell_shapes_gdf)},
        tables={
            "table": sd.models.TableModel.parse(
                adata,
                region_key="region",
                region="cell_boundaries",
                instance_key="cell_id"
            )
        },
    )
    sdata.write(f"{input_folder}/{sample}/raw_spatialdata.zarr", overwrite=True)

In [ ]:
# Step 2: wrap each sample into a SpatialData .zarr store
for folder_name in os.listdir(INPUT_DIR):
    print(folder_name)
    adata_to_spatialdata(folder_name, INPUT_DIR)
